# Diabetes — **Full Model Zoo (GPU‑aware)**
Dataset:
- **CSV_PATH:** `C:\Users\wlaeed\Desktop\projects\diabetes-prediction-main\diabetes_synthetic_1M.csv`
- **TARGET:** `diabetes_label`

Included models:
- **PyTorch MLP (GPU)** — AMP + `torch.compile` + fused AdamW + tqdm
- **XGBoost (GPU)**
- **LightGBM** (GPU if available)
- **CatBoost** (GPU if available)
- **RandomForest** (CPU)
- **ExtraTrees** (CPU)
- **Linear SVM** (CPU, downsampled)

Outputs a comparison table of **AUROC, AUPRC, Accuracy, F1, Time (s)**. No data leakage (train‑only scaling).


## Setup (run once in VS Code terminal)
```powershell
python -m venv venv
.
env\Scriptsctivate
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install pandas numpy scikit-learn tqdm
pip install xgboost
```


## Config

In [1]:
# === DATA & ZOO CONFIG ===
CSV_PATH   = r"C:\Users\wlaeed\Desktop\projects\diabetes-prediction-main\diabetes_synthetic_1M.csv"
TARGET     = "diabetes_label"
DROP_COLS  = []

SAMPLE_FRAC = None
NROWS       = None

VAL_SPLIT   = 0.2
RANDOM_STATE = 42

# MLP hyperparams (GPU)
BATCH_SIZE  = 1024
EPOCHS      = 10
LR          = 1e-3
MIXED_PRECISION = True
USE_COMPILE = True
NUM_WORKERS = None
PREFETCH    = 4

# CPU model caps
RF_MAX_ROWS   = 500_000
ET_MAX_ROWS   = 500_000
SVM_MAX_ROWS  = 200_000

SAVE_DIR    = "artifacts_zoo_full"
from pathlib import Path
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
print("Artifacts ->", SAVE_DIR)
print("CSV_PATH  ->", CSV_PATH)
print("TARGET    ->", TARGET)


Artifacts -> artifacts_zoo_full
CSV_PATH  -> C:\Users\wlaeed\Desktop\projects\diabetes-prediction-main\diabetes_synthetic_1M.csv
TARGET    -> diabetes_label


## Imports & GPU speed knobs

In [2]:
import os, json, time, warnings
from pathlib import Path
import numpy as np, pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
import torch._dynamo as dynamo
from torch._inductor import config as inductor_config

dynamo.config.suppress_errors = True      # fall back to eager instead of raising
inductor_config.max_autotune = False
inductor_config.max_autotune_gemm = False


_have_xgb = _have_lgbm = _have_cat = False
try:
    import xgboost as xgb; _have_xgb = True
except Exception as e:
    print("XGBoost not available:", e)
try:
    import lightgbm as lgb; _have_lgbm = True
except Exception:
    pass
try:
    import catboost as cat; _have_cat = True
except Exception:
    pass

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset
from torch.amp import autocast, GradScaler
from tqdm import tqdm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

warnings.filterwarnings("ignore")


Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


## Utilities

In [3]:
def coerce_labels(y: pd.Series) -> np.ndarray:
    if y.dtype.kind in "biufc":
        return (pd.to_numeric(y, errors="coerce").fillna(0).values != 0).astype(np.float32)
    lower = y.astype(str).str.lower().str.strip()
    mapping = {"yes":1,"y":1,"true":1,"1":1,"positive":1,"pos":1,"no":0,"n":0,"false":0,"0":0,"negative":0,"neg":0}
    mapped = lower.map(mapping).fillna(np.nan)
    if mapped.isna().any():
        codes, uniq = pd.factorize(lower, sort=True)
        if len(uniq) != 2:
            raise ValueError(f"Target has {len(uniq)} classes; need binary. Uniques={list(uniq)[:10]}")
        return (codes == codes.max()).astype(np.float32)
    return mapped.astype(np.float32).values

def build_features(df: pd.DataFrame, target: str, drop_cols):
    if target not in df.columns:
        raise ValueError(f"Target '{target}' not in CSV. Available (first 20): {list(df.columns)[:20]}")
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
    y = coerce_labels(df[target])
    X = df.drop(columns=[target])
    X = pd.get_dummies(X, drop_first=True).replace([np.inf,-np.inf], np.nan).astype(np.float32)
    X = X.fillna(X.mean(numeric_only=True))
    return X.values.astype(np.float32), y.astype(np.float32), X.columns.tolist()

def summarize_metrics(y_true, scores_or_proba, threshold=0.5):
    s = np.asarray(scores_or_proba).ravel().astype(np.float32)
    if s.min() < 0 or s.max() > 1:
        prob = 1.0 / (1.0 + np.exp(-s))
    else:
        prob = s
    pred = (prob >= threshold).astype(np.float32)
    acc = accuracy_score(y_true, pred)
    f1  = f1_score(y_true, pred, zero_division=0)
    try:   auroc = roc_auc_score(y_true, prob)
    except ValueError: auroc = float("nan")
    try:   auprc = average_precision_score(y_true, prob)
    except ValueError: auprc = float("nan")
    return {"acc":acc, "f1":f1, "auroc":float(auroc), "auprc":float(auprc)}


## Load data, split, scale (train‑only)

In [4]:
# Load CSV with optional sampling
if not Path(CSV_PATH).exists():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
df = pd.read_csv(CSV_PATH, nrows=NROWS)
if SAMPLE_FRAC is not None and 0 < SAMPLE_FRAC < 1.0:
    df = df.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE).reset_index(drop=True)
print("Rows:", len(df))

# Build features
X_raw, y, feature_cols = build_features(df, TARGET, DROP_COLS)

# Split first (avoid leakage), then scale using train stats only
Xtr_raw, Xval_raw, ytr, yval = train_test_split(X_raw, y, test_size=VAL_SPLIT, stratify=y, random_state=RANDOM_STATE)
scaler = StandardScaler()
Xtr = scaler.fit_transform(Xtr_raw); Xval = scaler.transform(Xval_raw)

# Tensors for MLP
Xtr_t = torch.from_numpy(Xtr).float(); ytr_t = torch.from_numpy(ytr).float()
Xval_t = torch.from_numpy(Xval).float(); yval_t = torch.from_numpy(yval).float()

# DataLoaders (fast)
if NUM_WORKERS is None:
    import os
    NUM_WORKERS = max(1, min(8, os.cpu_count()//2))
train_loader = DataLoader(
    TensorDataset(Xtr_t, ytr_t),
    batch_size=BATCH_SIZE, shuffle=True, pin_memory=(device.type=="cuda"),
    pin_memory_device="cuda" if device.type=="cuda" else "",
    num_workers=NUM_WORKERS, prefetch_factor=PREFETCH,
    persistent_workers=True, drop_last=True
)
val_loader = DataLoader(
    TensorDataset(Xval_t, yval_t),
    batch_size=BATCH_SIZE, shuffle=False, pin_memory=(device.type=="cuda"),
    pin_memory_device="cuda" if device.type=="cuda" else "",
    num_workers=max(1, NUM_WORKERS//2), prefetch_factor=max(1, PREFETCH//2),
    persistent_workers=True
)

Xtr_t.shape, Xval_t.shape, ytr_t.mean().item(), yval_t.mean().item(), NUM_WORKERS


Rows: 1000000


(torch.Size([800000, 8]),
 torch.Size([200000, 8]),
 0.3801549971103668,
 0.3801549971103668,
 8)

## Model 1 — PyTorch MLP (GPU)

In [5]:
class MLP(nn.Module):
    def __init__(self, in_features: int, hidden=(384, 192), dropout=0.10):
        super().__init__()
        layers, last = [], in_features
        for h in hidden:
            layers += [nn.Linear(last, h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers += [nn.Linear(last, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(1)

def evaluate_mlp(model: nn.Module, loader: DataLoader, device: torch.device) -> dict:
    model.eval()
    all_probs, all_y = [], []
    running = 0.0
    loss_fn = nn.BCEWithLogitsLoss()
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = loss_fn(logits, yb); running += loss.item() * xb.size(0)
            probs = torch.sigmoid(logits).float().cpu().numpy()
            all_probs.append(probs); all_y.append(yb.float().cpu().numpy())
    y_true = np.concatenate(all_y).astype(np.float32)
    prob   = np.concatenate(all_probs).astype(np.float32)
    pred = (prob >= 0.5).astype(np.float32)
    acc = accuracy_score(y_true, pred)
    f1  = f1_score(y_true, pred, zero_division=0)
    try:   auroc = roc_auc_score(y_true, prob)
    except ValueError: auroc = float("nan")
    try:   auprc = average_precision_score(y_true, prob)
    except ValueError: auprc = float("nan")
    return {"val_loss": running/len(y_true), "val_acc": acc, "val_f1": f1, "val_auroc": float(auroc), "val_auprc": float(auprc)}

model = MLP(in_features=Xtr_t.shape[1]).to(device)

if USE_COMPILE:
    try:
        model = torch.compile(model, mode="max-autotune")
        print("torch.compile: enabled")
    except Exception as e:
        print("torch.compile skipped:", e)

pos = max((ytr_t == 1).sum().item(), 1.0)
neg = max((ytr_t == 0).sum().item(), 1.0)
pos_weight = torch.tensor([neg/pos], device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

try:
    opt = optim.AdamW(model.parameters(), lr=LR, fused=True)
    print("Using fused AdamW")
except TypeError:
    opt = optim.AdamW(model.parameters(), lr=LR)
    print("Using standard AdamW")

scaler_amp = GradScaler("cuda", enabled=(MIXED_PRECISION and device.type=="cuda"))

best_mlp = {"epoch": -1, "metrics": {"val_auroc": -1.0}}
start = time.time()
for epoch in range(1, EPOCHS+1):
    model.train()
    running = seen = 0
    pbar = tqdm(train_loader, desc=f"MLP Epoch {epoch}/{EPOCHS}", ncols=100)
    for xb, yb in pbar:
        xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast(device_type="cuda", enabled=scaler_amp.is_enabled(), dtype=torch.float16):
            logits = model(xb)
            loss = loss_fn(logits, yb)
        scaler_amp.scale(loss).backward()
        scaler_amp.step(opt)
        scaler_amp.update()
        running += loss.item() * xb.size(0); seen += xb.size(0)
        pbar.set_postfix({"loss": f"{running/seen:.4f}"})
    metrics = evaluate_mlp(model, val_loader, device)
    print(f"val_loss={metrics['val_loss']:.4f}  acc={metrics['val_acc']:.4f}  f1={metrics['val_f1']:.4f}  auroc={metrics['val_auroc']:.4f}  auprc={metrics['val_auprc']:.4f}")
    if not np.isnan(metrics["val_auroc"]) and metrics["val_auroc"] > best_mlp["metrics"].get("val_auroc", -1):
        best_mlp = {"epoch": epoch, "metrics": metrics.copy()}
        ckpt = {"kind":"mlp","state_dict":model.state_dict(),"feature_cols":list(feature_cols),
                "scaler_mean": getattr(globals().get("scaler"), "mean_", None),
                "scaler_scale": getattr(globals().get("scaler"), "scale_", None),
                "target": TARGET}
        torch.save(ckpt, Path(SAVE_DIR)/"mlp_model.pt")
mlp_time = time.time() - start
print(f"MLP done in {mlp_time:.1f}s. Best epoch {best_mlp['epoch']}, AUROC={best_mlp['metrics'].get('val_auroc', float('nan')):.4f}")


torch.compile: enabled
Using fused AdamW


MLP Epoch 1/10:   0%|                                                       | 0/781 [00:00<?, ?it/s]W0818 00:43:24.157912 23824 Lib\site-packages\torch\_inductor\utils.py:1048] [0/0] Not enough SMs to use max_autotune_gemm mode
W0818 00:43:24.205811 23824 Lib\site-packages\torch\_dynamo\convert_frame.py:1125] WON'T CONVERT forward C:\Users\wlaeed\AppData\Local\Temp\ipykernel_23824\2687572835.py line 10 
W0818 00:43:24.205811 23824 Lib\site-packages\torch\_dynamo\convert_frame.py:1125] due to: 
W0818 00:43:24.205811 23824 Lib\site-packages\torch\_dynamo\convert_frame.py:1125] Traceback (most recent call last):
W0818 00:43:24.205811 23824 Lib\site-packages\torch\_dynamo\convert_frame.py:1125]   File "c:\Users\wlaeed\Desktop\projects\diabetes-prediction-main\.venv\lib\site-packages\torch\_dynamo\output_graph.py", line 1446, in _call_user_compiler
W0818 00:43:24.205811 23824 Lib\site-packages\torch\_dynamo\convert_frame.py:1125]     compiled_fn = compiler_fn(gm, self.example_inputs())
W081

val_loss=0.0278  acc=0.9900  f1=0.9870  auroc=0.9996  auprc=0.9995


MLP Epoch 2/10: 100%|███████████████████████████████| 781/781 [00:04<00:00, 156.60it/s, loss=0.0362]


val_loss=0.0166  acc=0.9952  f1=0.9937  auroc=0.9999  auprc=0.9998


...

val_loss=0.0073  acc=0.9977  f1=0.9970  auroc=0.9999  auprc=0.9999


MLP Epoch 9/10: 100%|███████████████████████████████| 781/781 [00:05<00:00, 133.74it/s, loss=0.0114]


val_loss=0.0072  acc=0.9977  f1=0.9970  auroc=0.9999  auprc=0.9999


MLP Epoch 10/10: 100%|██████████████████████████████| 781/781 [00:05<00:00, 151.72it/s, loss=0.0109]


val_loss=0.0081  acc=0.9971  f1=0.9962  auroc=0.9999  auprc=0.9999
MLP done in 67.7s. Best epoch 8, AUROC=0.9999


## Model 2 — XGBoost (GPU)

In [6]:
xgb_result = None
if _have_xgb:
    dtrain = xgb.DMatrix(Xtr, label=ytr)
    dval   = xgb.DMatrix(Xval, label=yval)
    num_round = 1000
    params = {
        "objective": "binary:logistic",
        "eval_metric": ["auc","aucpr"],
        "max_depth": 8,
        "eta": 0.08,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "tree_method": "hist",
        "device": "cuda" if device.type=="cuda" else "cpu",
    }
    try:
        from xgboost.callback import TrainingCallback
        class TQDMCallback(TrainingCallback):
            def __init__(self, total): self.pbar = tqdm(total=total, desc="XGBoost", ncols=100)
            def after_iteration(self, model, epoch, evals_log): self.pbar.update(1); return False
            def __del__(self):
                try: self.pbar.close()
                except: pass
        callbacks=[TQDMCallback(num_round)]
    except Exception:
        callbacks=[]
    start = time.time()
    booster = xgb.train(params, dtrain, num_boost_round=num_round,
                        evals=[(dtrain,"train"),(dval,"valid")],
                        early_stopping_rounds=100, verbose_eval=False,
                        callbacks=callbacks)
    xgb_time = time.time() - start
    xgb_result = {"epoch": booster.best_iteration, "auc": booster.best_score, "time_s": xgb_time}
    booster.save_model(str(Path(SAVE_DIR)/"xgb_model.json"))
    print("XGBoost best:", xgb_result)
else:
    print("Skipping XGBoost (package not installed)")


XGBoost:  17%|████████▌                                          | 167/1000 [00:02<00:11, 72.48it/s]

XGBoost best: {'epoch': 72, 'auc': 0.9999446274189401, 'time_s': 2.807966470718384}


## Model 3 — LightGBM (GPU if available)

In [7]:
lgb_result = None
if _have_lgbm:
    train_ds = lgb.Dataset(Xtr, label=ytr)
    valid_ds = lgb.Dataset(Xval, label=yval, reference=train_ds)
    params = {
        "objective": "binary",
        "metric": ["auc", "average_precision"],
        "learning_rate": 0.08,
        "num_leaves": 255,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.9,
        "bagging_freq": 1,
        "min_data_in_leaf": 50,
        "device": "gpu" if device.type=="cuda" else "cpu",
    }
    start = time.time()
    booster = lgb.train(
        params,
        train_ds,
        num_boost_round=3000,
        valid_sets=[train_ds, valid_ds],
        valid_names=["train","valid"],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(50)],
    )
    lgb_time = time.time() - start
    best_iter = booster.best_iteration
    evals = booster.best_score
    auc = evals.get("valid", {}).get("auc", float("nan"))
    auprc = evals.get("valid", {}).get("average_precision", float("nan"))
    lgb_result = {"epoch": best_iter, "auc": auc, "auprc": auprc, "time_s": lgb_time}
    booster.save_model(str(Path(SAVE_DIR)/"lgb_model.txt"))
    print("LightGBM best:", lgb_result)
else:
    print("Skipping LightGBM (package not installed)")


Skipping LightGBM (package not installed)


## Model 4 — CatBoost (GPU if available)

In [8]:
cat_result = None
if _have_cat:
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "learning_rate": 0.08,
        "depth": 8,
        "l2_leaf_reg": 3.0,
        "iterations": 3000,
        "random_seed": 42,
        "task_type": "GPU" if device.type=="cuda" else "CPU",
        "devices": "0",
        "verbose": 200,
        "od_type": "Iter",
        "od_wait": 200,
    }
    train_pool = cat.Pool(Xtr, ytr)
    valid_pool = cat.Pool(Xval, yval)
    start = time.time()
    model_cat = cat.CatBoostClassifier(**params)
    model_cat.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=200)
    cat_time = time.time() - start
    yprob = model_cat.predict_proba(Xval)[:,1]
    metrics = summarize_metrics(yval, yprob)
    metrics["time_s"] = cat_time
    cat_result = metrics
    model_cat.save_model(str(Path(SAVE_DIR)/"cat_model.cbm"))
    print("CatBoost:", metrics)
else:
    print("Skipping CatBoost (package not installed)")


Skipping CatBoost (package not installed)


## Model 5 — RandomForest (CPU, optional sub-sample)

In [9]:
rf_result = None
rows = Xtr.shape[0]
idx = np.arange(rows)
if rows > RF_MAX_ROWS:
    np.random.seed(RANDOM_STATE)
    sub = np.random.choice(idx, size=RF_MAX_ROWS, replace=False)
    Xtr_rf, ytr_rf = Xtr[sub], ytr[sub]
    print(f"RandomForest: downsampled train to {len(sub)}/{rows}")
else:
    Xtr_rf, ytr_rf = Xtr, ytr

start = time.time()
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
)
rf.fit(Xtr_rf, ytr_rf)
yprob = rf.predict_proba(Xval)[:,1]
rf_time = time.time() - start
rf_result = summarize_metrics(yval, yprob); rf_result["time_s"] = rf_time
print("RandomForest:", rf_result)


RandomForest: downsampled train to 500000/800000


XGBoost:  17%|████████▊                                          | 173/1000 [00:22<00:11, 72.48it/s]

RandomForest: {'acc': 0.999515, 'f1': 0.9993617035277396, 'auroc': 0.9999280946441075, 'auprc': 0.9998952990162507, 'time_s': 40.08619976043701}


## Model 6 — ExtraTrees (CPU, optional sub-sample)

In [10]:
et_result = None
rows = Xtr.shape[0]
idx = np.arange(rows)
if rows > ET_MAX_ROWS:
    np.random.seed(RANDOM_STATE+1)
    sub = np.random.choice(idx, size=ET_MAX_ROWS, replace=False)
    Xtr_et, ytr_et = Xtr[sub], ytr[sub]
    print(f"ExtraTrees: downsampled train to {len(sub)}/{rows}")
else:
    Xtr_et, ytr_et = Xtr, ytr

start = time.time()
et = ExtraTreesClassifier(
    n_estimators=400,
    max_depth=None,
    n_jobs=-1,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)
et.fit(Xtr_et, ytr_et)
yprob = et.predict_proba(Xval)[:,1]
et_time = time.time() - start
et_result = summarize_metrics(yval, yprob); et_result["time_s"] = et_time
print("ExtraTrees:", et_result)


ExtraTrees: downsampled train to 500000/800000
ExtraTrees: {'acc': 0.997415, 'f1': 0.9965958399452174, 'auroc': 0.9996997532341523, 'auprc': 0.9996640711564111, 'time_s': 29.633283138275146}


## Model 7 — Linear SVM (CPU, downsampled)

In [11]:
svm_result = None
rows = Xtr.shape[0]
idx = np.arange(rows)
cap = min(SVM_MAX_ROWS, rows)
np.random.seed(RANDOM_STATE+2)
sub = np.random.choice(idx, size=cap, replace=False)
Xtr_svm, ytr_svm = Xtr[sub], ytr[sub]
print(f"LinearSVC: using {len(sub)}/{rows} rows")

start = time.time()
svm = LinearSVC(
    C=1.0,
    class_weight="balanced",
    max_iter=5000,
    dual=True,
)
svm.fit(Xtr_svm, ytr_svm)
scores = svm.decision_function(Xval)
svm_time = time.time() - start
svm_result = summarize_metrics(yval, scores); svm_result["time_s"] = svm_time
print("LinearSVC:", svm_result)


LinearSVC: using 200000/800000 rows
LinearSVC: {'acc': 0.880345, 'f1': 0.8500272609341414, 'auroc': 0.9580525393686237, 'auprc': 0.9465956634994721, 'time_s': 19.22149634361267}


## Compare models

In [12]:
rows = []

rows.append({
    "model": "MLP (PyTorch GPU)",
    "auroc": best_mlp["metrics"].get("val_auroc", np.nan),
    "auprc": best_mlp["metrics"].get("val_auprc", np.nan),
    "acc":   best_mlp["metrics"].get("val_acc", np.nan),
    "f1":    best_mlp["metrics"].get("val_f1", np.nan),
    "time_s": np.nan if 'mlp_time' not in globals() else mlp_time,
})

if _have_xgb and 'xgb_result' in globals() and xgb_result is not None:
    rows.append({"model": "XGBoost (GPU)", "auroc": xgb_result["auc"], "auprc": np.nan, "acc": np.nan, "f1": np.nan, "time_s": xgb_result["time_s"]})

if _have_lgbm and 'lgb_result' in globals() and lgb_result is not None:
    rows.append({"model": "LightGBM", "auroc": lgb_result["auc"], "auprc": lgb_result["auprc"], "acc": np.nan, "f1": np.nan, "time_s": lgb_result["time_s"]})

if _have_cat and 'cat_result' in globals() and cat_result is not None:
    rows.append({"model": "CatBoost", "auroc": cat_result["auroc"], "auprc": cat_result["auprc"], "acc": cat_result["acc"], "f1": cat_result["f1"], "time_s": cat_result["time_s"]})

if 'rf_result' in globals() and rf_result is not None:
    rows.append({"model": "RandomForest", **rf_result})

if 'et_result' in globals() and et_result is not None:
    rows.append({"model": "ExtraTrees", **et_result})

if 'svm_result' in globals() and svm_result is not None:
    rows.append({"model": "LinearSVM", **svm_result})

import pandas as pd
df_results = pd.DataFrame(rows)
df_results = df_results[["model","auroc","auprc","acc","f1","time_s"]]
df_results.sort_values(by="auroc", ascending=False, inplace=True)
df_results.reset_index(drop=True, inplace=True)
df_results


,model,auroc,auprc,acc,f1,time_s
0,MLP (PyTorch GPU),0.999945,0.999916,0.997730,0.997016,67.665283
1,XGBoost (GPU),0.999945,NaN,NaN,NaN,2.807966
2,RandomForest,0.999928,0.999895,0.999515,0.999362,40.086200
3,ExtraTrees,0.999700,0.999664,0.997415,0.996596,29.633283
4,LinearSVM,0.958053,0.946596,0.880345,0.850027,19.221496
